# Lake Mead water extent and shoreline elevation

Sentinel-1 maps observed water. Google GEM-15's bare-earth DTM supplies land elevations for an indirect water-level estimate, in metres EGM96. US Bureau of Reclamation gauge readings use USGS 1912. In the absence of an independent datum transform, treat raw residuals as cross-datum differences.

This dataset is produced by Google. [Earth Engine catalog](https://developers.google.com/earth-engine/datasets/catalog/projects_climate-and-sustainability_assets_gem-15_v1)

In [ ]:
from datetime import date
from pathlib import Path

from IPython.display import display
import pandas as pd

from lake_mead import gee, inundation, observations, sentinel1, validation, visualization
from lake_mead.config import Settings
from lake_mead.workflow import Analysis

start, end = date(2022, 7, 1), date(2022, 9, 1)
settings = Settings(orbit_pass="DESCENDING", polarization="VV")
root = Path(".") if Path("pyproject.toml").is_file() else Path("..")
output = root / "outputs/notebook"
output.mkdir(parents=True, exist_ok=True)
gee.initialize()
monitor_window = None  # Set "overton" to restrict area and elevation to that window.
analysis = Analysis.create(settings, monitor_window=monitor_window)

## Acquisitions, samples and quality control

The analysis holds relative orbit fixed after acquisition discovery. Map comparison requires two QC-passing acquisitions. The map uses the smallest and largest reported areas by default. Set selected_indices to two explicit acquisition indices to choose a different pair. Widen the date range to span a larger pool change, then hold track fixed across runs.

In [ ]:
areas, estimates, samples = analysis.run(start, end)
display(areas)
if not estimates.empty:
    display(estimates.reindex(columns=[
        "date", "domain", "qc", "samples", "retained", "blocks", "block_iqr_m",
        "elevation_m", "lower_quartile_m", "valid_terrain_fraction",
        "screened_elevation_m", "screened_samples", "screened_qc",
    ]))
areas.to_csv(output / "acquisitions.csv", index=False)
estimates.to_csv(output / "elevations.csv", index=False)
samples.to_csv(output / "shoreline_samples.csv", index=False)

## Observed water, terrain gaps and modeled land gain

Inspect sample clusters, the missing-terrain layer and DSM variability together. High roughness or a large DSM–DTM gap may be real relief or an artifact. A fixed screening experiment tracks how these proxies affect estimates and sample coverage. Modeled gain illustrates a level above the earlier shoreline estimate where valid low land connects to observed water. The scenario retains existing water as a seed and leaves masked terrain unresolved.

In [ ]:
valid = areas.loc[areas["qc"].eq("pass")].sort_values("date")
if valid.empty:
    raise RuntimeError("No acquisitions passed QC. Inspect the acquisitions table and date filters.")
selected_indices = None  # Set a list of two QC-passing indices to override.
if selected_indices:
    valid = valid.set_index("index", drop=False).loc[selected_indices].sort_values("date")
elif len(valid) > 1:
    extremes = [valid["water_m2"].idxmin(), valid["water_m2"].idxmax()]
    if extremes[0] != extremes[1]:
        valid = valid.loc[extremes].sort_values("date")
earlier_index = valid.iloc[0]["index"]
later_index = valid.iloc[-1]["index"]
inventory = sentinel1.scenes(start, end, analysis.bounds, settings)
by_index = {scene.index: scene for scene in inventory}
earlier = analysis.classify(by_index[earlier_index])
later = analysis.classify(by_index[later_index]) if later_index != earlier_index else None
shown_index = later_index if later is not None else earlier_index
shown = samples.loc[samples["index"].eq(shown_index)] if not samples.empty else samples
modeled = None
if not estimates.empty and "elevation_m" in estimates:
    initial = estimates.loc[estimates["index"].eq(earlier_index) & estimates["qc"].eq("pass")]
    if not initial.empty:
        baseline_m = float(initial["elevation_m"].median())
        modeled = inundation.scenario(earlier, analysis.terrain, baseline_m + 3,
                                      analysis.corridor, analysis.bounds, settings)
lake_map = visualization.interactive(analysis, earlier, later, shown, modeled)
display(lake_map)
threshold_panels = [(str(valid.iloc[0]["date"]), earlier)]
if later is not None:
    threshold_panels.append((str(valid.iloc[-1]["date"]), later))
visualization.plot_thresholds(threshold_panels, output).show()
if later is not None:
    change_area = inundation.change_areas(earlier, later, analysis.bounds, settings)
    change_table = pd.DataFrame([{
        "earlier_date": valid.iloc[0]["date"], "later_date": valid.iloc[-1]["date"], **change_area
    }])
    display(change_table)
    change_table.to_csv(output / "observed_change_area.csv", index=False)
    labels = (str(valid.iloc[0]["date"]), str(valid.iloc[-1]["date"]))
    visualization.export_map(analysis, earlier, later, output, shown, labels).show()

## Independent gauge comparison

The acquisition timestamp is converted to fixed Mountain Standard Time and joined to a daily Reclamation measurement on the same local date. Read `docs/methods.md` before interpreting MAE, RMSE or bias. The paired-screen baseline reports metrics on the same dates as the roughness-screened estimate.

In [ ]:
gauge = observations.download(min(start, date.fromisoformat(areas["date"].min())), end)
matched = pd.DataFrame()
if "elevation_m" in estimates:
    matched, scores = validation.compare(estimates, gauge)
    display(pd.DataFrame(scores))
    matched.to_csv(output / "validation_pairs.csv", index=False)
figures = visualization.plot_results(areas, estimates, matched, output)
for figure in figures.values():
    figure.show()